In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
import IPython.display as ipd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf
import librosa
import params

## Data

Load data from cache

In [ ]:
df = pd.read_csv("data.csv")
audio = np.load("audio.npy")

Manipulate data

In [ ]:
unique_genres = sorted(list(set(df["genre"])))
n_genres = len(unique_genres)
assert n_genres == 8
genre_to_id = {unique_genres[_]: _ for _ in range(n_genres)}
id_to_genre = {_: unique_genres[_] for _ in range(n_genres)}
df["genre_id"] = df["genre"].map(genre_to_id)

Split the dataset

In [ ]:
df_test = df[: int(params.test_size * len(df))].reset_index(drop=True)
df_val = df[
    int(params.test_size * len(df)) : int(
        (params.test_size + params.val_size) * len(df)
    )
].reset_index(drop=True)
df_train = df[int((params.test_size + params.val_size) * len(df)) :].reset_index(
    drop=True
)


audio_train = audio[int((params.test_size + params.val_size) * len(audio)) :]
audio_test = audio[: int(params.test_size * len(audio))]
audio_val = audio[
    int(params.test_size * len(audio)) : int(
        (params.test_size + params.val_size) * len(audio)
    )
]

assert len(df_train) == len(audio_train)
assert len(df_test) == len(audio_test)
assert len(df_val) == len(audio_val)
assert len(df_train) + len(df_val) + len(df_test) == len(df)

print(f"Train size: {len(df_train)}")
print(f"Test size: {len(df_test)}")
print(f"Validation size: {len(df_val)}")

Utilization Functions

In [ ]:
def compute_mel_spectrogram(y: np.ndarray):
    mel_spectrogram = librosa.feature.mfcc(y=y, sr=params.sample_rate, n_mels=40)
    mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)
    return mel_spectrogram_db


def compute_spectral_centroid(y: np.ndarray):
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=params.sample_rate)
    return spectral_centroid


def compute_spectral_bandwidth(y: np.ndarray):
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=params.sample_rate)
    return spectral_bandwidth


def compute_spectral_rolloff(y: np.ndarray):
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=params.sample_rate)
    return spectral_rolloff


def compute_spectral_contrast(y: np.ndarray):
    spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=params.sample_rate)
    return spectral_contrast


def compute_spectral_flatness(y: np.ndarray):
    spectral_flatness = librosa.feature.spectral_flatness(y=y)
    return spectral_flatness


def compute_spectral_features(y: np.ndarray):
    features = []
    features.append(compute_spectral_centroid(y))
    features.append(compute_spectral_bandwidth(y))
    features.append(compute_spectral_rolloff(y))
    features.append(compute_spectral_contrast(y))
    features.append(compute_spectral_flatness(y))
    return np.vstack(features).flatten()


n_features = len(compute_spectral_features(audio[0]))
assert isinstance(n_features, int)
assert n_features > 1000


def show_mel_spectrogram(index: int):
    mel_spectrogram = compute_mel_spectrogram(audio[index])

    plt.figure(figsize=(10, 4))
    librosa.display.specshow(
        mel_spectrogram,
        sr=params.sample_rate,
        x_axis="time",
        y_axis="mel",
        cmap="coolwarm",
    )
    plt.colorbar(format="%+2.0f dB")
    plt.tight_layout()
    plt.title(
        f'Mel Spectrogram for audio at index {index}, genre: {df.iloc[index]["genre"]}'
    )
    plt.show()


def play_audio(index: int):
    ipd.display(ipd.Audio(data=audio[index], rate=params.sample_rate))


def investigate(index: int):
    show_mel_spectrogram(index)
    play_audio(index)

In [ ]:
investigate(np.random.randint(0, len(audio) - 1))

## Model

In [ ]:
scaler = StandardScaler().fit(audio_train)

In [ ]:
def compute_spectral_features_for_dataset(audio_dataset):
    return np.array(
        [
            compute_spectral_features(audio_dataset[i])
            for i in tqdm(range(len(audio_dataset)))
        ]
    )


X_train = compute_spectral_features_for_dataset(audio_train)
X_test = compute_spectral_features_for_dataset(audio_test)
X_val = compute_spectral_features_for_dataset(audio_val)

In [ ]:
y_train = df_train["genre_id"].values
y_test = df_test["genre_id"].values
y_val = df_val["genre_id"].values

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output
import tensorflow as tf


class LivePlot(tf.keras.callbacks.Callback):
    def __init__(self, logy=False):
        super().__init__()
        self.logy = logy
        self.train_loss = []
        self.val_loss = []
        self.train_acc = []
        self.val_acc = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        # Collect metrics
        self.train_loss.append(logs.get("loss"))
        self.val_loss.append(logs.get("val_loss"))
        self.train_acc.append(logs.get("accuracy"))
        self.val_acc.append(logs.get("val_accuracy"))

        # Update plot
        clear_output(wait=True)
        plt.figure(figsize=(12, 5))

        # loss
        plt.subplot(1, 2, 1)
        plt.plot(self.train_loss, label="Train Loss")
        plt.plot(self.val_loss, label="Validation Loss")
        plt.title("Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss"+ " (Log Scale)" if self.logy else "Loss")
        if self.logy:
            plt.yscale("log")
        plt.legend()
        plt.grid(True, which="both")

        # accuracy
        plt.subplot(1, 2, 2)
        plt.plot(self.train_acc, label="Train Accuracy")
        plt.plot(self.val_acc, label="Validation Accuracy")
        plt.axhline(
            y=1 / n_genres, color="red", linestyle="--", linewidth=2, label="Random Threshold"
        )
        plt.title("Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.show()

In [ ]:
def create_model():
    """Simple MLP model"""
    inp = tf.keras.layers.Input(shape=(n_features,))
    x = tf.keras.layers.Dense(256, activation="relu")(inp)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_genres, activation="softmax")(x)
    model = tf.keras.Model(inputs=inp, outputs=out)
    return model

In [ ]:
model = create_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],  # loss is implicityly included
)
model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=500,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=10, restore_best_weights=True
        ),
        LivePlot(logy=False),
    ],
    verbose=0,
)